## Read the data

In [1]:
import numpy as np
import pandas as pd
import sys
PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)
METHOD = "hmm_pomegranate"

In [2]:
DATA_PATH = 'D:/Google Drive/Work/ITMO/data/length_dependent_hmm/per_length_per_kfold_split/' # windows
# DATA_PATH = '/home/denklewer/Data/length_dependent_hmm/' #ubuntu
TARGET_PATH_TO_RESULTS = "experiment_results/retrained_models/"
RUNS = 15
TOTAL_SPLITS = 1

# Read models

In [3]:
MODELS_DESCRIPTIONS = {
    "viterbi_simple_with_pseudo_and_weights_7": {
        "path": "experiment_results/core_finding_models_viterbi_simple_with_pseudo/",
        "states": 7,
        "params_string": "30-min_iters",
        "combination_size": 1,
        "alleles": ['HLA-DRB1*01:01',
                    # 'HLA-DPA1*01:03-DPB1*04:01'
                    # 'HLA-A*02:01'
                    ],
        "target_lengths": [12, 13, 14, 15, 16, 17, 18],
        "core_start_state": 7,
        "core_end_state": 8

    },
    "viterbi_with_pseudo_and_weights_7": {
        "path": "experiment_results/core_finding_models_viterbi_simple_with_pseudo/",
        "states": 35,
        "params_string": "30-min_iters",
        "combination_size": 1,
        "alleles": ['HLA-DRB1*01:01',
                    # 'HLA-DPA1*01:03-DPB1*04:01'
                    # 'HLA-A*02:01'
                    ],
        "target_lengths": [12, 13, 14, 15, 16, 17, 18],
        "core_start_state": 35,
        "core_end_state": 36,
        "need_to_delete": [37, 38]

    },
    "viterbi_multiple_7": {
        "path": "experiment_results/core_multiple_allele/",
        "states": 35,
        "params_string": "30-min_iters",
        "combination_size": 2,
        "alleles": ['HLA-DRB1*01:01',
                    'HLA-DPA1*01:03/DPB1*04:01'
                    # 'HLA-A*02:01'
                    ],
        "target_lengths": [12, 13, 14, 15, 16, 17, 18],
        "core_start_state": [35, 36],
        "core_end_state": [37, 38],
        "need_to_delete": [39, 40, 41, 42]

    }



    # "mix_of_alleles_balanced":
    # {
    #     "alleles": ['HLA-A*01:01', 'HLA-A*03:01', 'HLA-A*02:01'],
    #     "path": "experiment_results/alleles_with_groups_balanced/",
    #     "states": 25,
    #     "params_string": "5-min_iters",
    #     "target_lengths": [9, 10]
    # }
}

In [4]:
selected_models = [
     "viterbi_multiple_7"
]
models_to_read = ["v"]

In [5]:
from hmm_logic_methods import load_model

In [6]:
import itertools

name_to_run = dict()
name_to_split = dict()
name_to_type = dict()
name_to_model_root = dict()
name_to_target_lengths = dict()
selected_binding_models = list()
per_name_models = dict()


def populate_models_infos(model_name, run_num, split_num, model_path, model_type, target_lengths ):
    selected_binding_models.append(model_name)

    name_to_run[model_name] = run_num
    name_to_split[model_name] = split_num
    name_to_model_root[model_name] = model_path
    assert model_type in ["bw", "v", "bwv", "bw-it", "bwv-it", "v-it"] or "mixed" in model_type
    name_to_type[model_name] = model_type
    name_to_target_lengths[model_name] = target_lengths

for model in selected_models:
    model_description = MODELS_DESCRIPTIONS[model]
    combination_size = model_description['combination_size']
    model_alleles = model_description['alleles']
    MODELS_PATH = model_description['path']
    params_string = model_description['params_string']
    states = model_description['states']
    TARGET_LENGTHS = model_description['target_lengths']
    target_length_string = "-".join([str(l) for l in TARGET_LENGTHS])
    MIXED_ALLELES = list()

    for combination in itertools.combinations(sorted(model_alleles), combination_size):
        allele_name = "+".join(combination)
        # allele_name = f"{allele_name1}+{allele_name2}"
        MIXED_ALLELES.append(allele_name)

    for allele_name in MIXED_ALLELES:
        target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_').replace('/', '_')
        for run_num_b in range(RUNS):
            for split_num_b in range(TOTAL_SPLITS):
                #bw
                if 'bw' in models_to_read:
                    bw_model_name = f"{run_num_b}_run-{target_length_string}-{states}-st-baum-welch-alg-{params_string}_model-{target_allele_name}-{split_num_b}"
                    print(bw_model_name)
                    populate_models_infos(bw_model_name, run_num_b,split_num_b, MODELS_PATH, f"bw-it-mixed({target_allele_name})", TARGET_LENGTHS)
                if 'bwv' in models_to_read:
                    #bwv
                    bwv_model_name = f"{run_num_b}_run-{target_length_string}-{states}-st-baum-welch-alg-{params_string}_model-{target_allele_name}-{split_num_b}_plus_viterbi"
                    populate_models_infos(bwv_model_name, run_num_b,split_num_b, MODELS_PATH, f"bwv-it-mixed({target_allele_name})", TARGET_LENGTHS)
                if 'v' in models_to_read:
                    #v
                    v_model_name = f"{run_num_b}_run-{target_length_string}-{states}-st-viterbi-alg-{params_string}_model-{target_allele_name}-{split_num_b}"
                    populate_models_infos(v_model_name, run_num_b,split_num_b, MODELS_PATH, f"v-it-mixed({target_allele_name})", TARGET_LENGTHS)



for model_name in selected_binding_models:
    per_name_models[model_name] =  load_model(name_to_model_root[model_name], model_name)

In [7]:
per_name_models.keys()

dict_keys(['0_run-12-13-14-15-16-17-18-35-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0', '1_run-12-13-14-15-16-17-18-35-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0', '2_run-12-13-14-15-16-17-18-35-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0', '3_run-12-13-14-15-16-17-18-35-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0', '4_run-12-13-14-15-16-17-18-35-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0', '5_run-12-13-14-15-16-17-18-35-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0', '6_run-12-13-14-15-16-17-18-35-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0', '7_run-12-13-14-15-16-17-18-35-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0', '8_run-12-13-14-15-16-17-18-35-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0',

## Read data

In [8]:
DATA_PATH_BASE = 'G:/My Drive/Work/ITMO/data'
# DATA_PATH_BASE = 'D:/Google Drive/Work/ITMO/data' #windows
# DATA_PATH_BASE = '/home/denklewer/Data' #ubuntu

DATA_PATH = f'{DATA_PATH_BASE}/length_dependent_hmm/peptide_cores' # work laptop
data_df = pd.read_csv(f"{DATA_PATH}/extractd_cores.csv", index_col=0, sep=';')
USE_WEIGHTS = False

In [9]:
per_allele_per_kfold_per_length_binders_train =dict()
per_allele_per_kfold_per_length_binders_test =dict()
ALLELES = list(data_df.allele.unique())
TARGET_LENGTHS = [9]
for allele_name in ALLELES:
    per_allele_per_kfold_per_length_binders_train[allele_name] = dict()
    per_allele_per_kfold_per_length_binders_test[allele_name] = dict()
    for split_num in range(TOTAL_SPLITS):
        per_allele_per_kfold_per_length_binders_train[allele_name][split_num] = dict()
        per_allele_per_kfold_per_length_binders_test[allele_name][split_num] = dict()
        for target_length in TARGET_LENGTHS:
            per_allele_per_kfold_per_length_binders_train[allele_name][split_num][target_length]= \
            data_df[(data_df.allele == allele_name) & (data_df.core_len == target_length)]['core_peptide'].values
            per_allele_per_kfold_per_length_binders_test[allele_name][split_num][target_length]= \
            data_df[(data_df.allele == allele_name) & (data_df.core_len == target_length)]['core_peptide'].values[:50]

In [10]:
from data_reading_methods import merge_per_allele_sets_into_combination_sets
check_peptides = True
combination_size = 2
per_allele_per_kfold_per_length_binders_train_old = per_allele_per_kfold_per_length_binders_train
per_allele_per_kfold_per_length_binders_test_old = per_allele_per_kfold_per_length_binders_test


if check_peptides:
    transform_dicts=None

per_allele_per_kfold_per_length_binders_train, per_allele_per_kfold_per_length_binders_test = merge_per_allele_sets_into_combination_sets(
    per_allele_per_kfold_per_length_binders_train, per_allele_per_kfold_per_length_binders_test,
    combination_size=combination_size,
    selected_alelles=ALLELES,
    total_splits=TOTAL_SPLITS,
    list_of_transform_dicts=None
)
OLD_ALLELES = ALLELES
NEW_ALLELES = list(per_allele_per_kfold_per_length_binders_test.keys())
ALLELES = NEW_ALLELES

HLA-DPA1*01:03/DPB1*04:01+HLA-DRB1*01:01
Length 9: HLA-DPA1*01:03/DPB1*04:01 - 7075 HLA-DRB1*01:01 - 7068 


In [11]:
ALLELES

['HLA-DPA1*01:03/DPB1*04:01+HLA-DRB1*01:01']

In [71]:
from data_reading_methods import extract_count_df, calculate_weights_based_on_length_counts

count_df_train = extract_count_df(per_allele_per_kfold_per_length_binders_train_old,
                                  selected_alleles=OLD_ALLELES,
                                  total_splits=TOTAL_SPLITS)
count_df_test = extract_count_df(per_allele_per_kfold_per_length_binders_test_old,
                                 selected_alleles=OLD_ALLELES,
                                 total_splits=TOTAL_SPLITS)

In [72]:
per_allele_per_kfold_per_length_weights = calculate_weights_based_on_length_counts(count_df_train,
                                                                                   selected_alleles=ALLELES,
                                                                                   total_splits=TOTAL_SPLITS,
                                                                                   selected_lengths=TARGET_LENGTHS,
                                                                                   use_weights=USE_WEIGHTS)

Total data length for split 0 is 14143
Weights for lentgh 9 are {'HLA-DPA1*01:03/DPB1*04:01': 1, 'HLA-DRB1*01:01': 1}


## Training params

In [73]:
stop_threshold = 0
cutoff = 0.00005
per_symbol_portion = -5
for target_length in TARGET_LENGTHS:
    target_length_for_training = target_length
    train_size = len(per_allele_per_kfold_per_length_binders_train[ALLELES[0]][0][target_length])
    ## Stop threshold Hardcoded here
    stop_threshold +=  abs(cutoff * train_size * per_symbol_portion  * target_length_for_training)
stop_threshold = stop_threshold / len(TARGET_LENGTHS)
stop_threshold

31.82175

In [74]:
# Training params
algorithm = 'viterbi'

params = dict()

# params['st'] = 7
# params['multiple_check_input'] = False
# params['alg'] = algorithm
# params['min_iters'] = 30
# params['max_iters'] = 65
# params['stop_threshold'] = stop_threshold
# params['emission_pseudocount'] = 1
# params['transition_pseudocount'] = 0
# params['verbose'] = True
# params['lr_decay'] = 0
# params['edge_inertia'] = 0.25
# params['distribution_inertia'] = 0
# params['groups'] = 7
# params['states_per_group'] = 1
# params['use_pseudocount'] = True

params['st'] = 20
params['multiple_check_input'] = False
params['alg'] = algorithm
params['min_iters'] = 30
params['max_iters'] = 65
params['stop_threshold'] = stop_threshold
params['emission_pseudocount'] = 1
params['transition_pseudocount'] = 1
params['verbose'] = True
params['lr_decay'] = 0
params['edge_inertia'] = 0
params['distribution_inertia'] = 0.25
params['groups'] = 4
params['states_per_group'] = 5
params['use_pseudocount'] = False
# Class 1 params
params['self_cycle'] = False
params['start_cycle'] = False
params['end_cycle'] = False

params['freeze_distr'] = False  # wether to freeze start and end cycle distributions
params['cycle_pseudocount'] = 1

assert params['st'] % params['groups'] == 0
assert params['st'] % params['states_per_group'] == 0
assert params['st'] == params['groups'] * params['states_per_group']

params_for_string = {key: value for key, value in params.items() if key in ['st', 'alg', 'min_iters']}
params_string = "-".join([f"{value}-{key}" for key, value in params_for_string.items()])
lengths_string = "-".join(map(str, TARGET_LENGTHS))

initial_params = None

## Initialize new models here

In [75]:
from pomegranate import *
import json
import numpy as np
import pandas as pd
from pomegranate.io import BatchedDataGenerator, SequenceGenerator
from hmm_visualization_methods import convert_graph_to_good_format
import networkx as nx
from tqdm import tqdm
from collections import defaultdict
from pomegranate import IndependentComponentsDistribution

MINUS_INF_CONSTANT = -20
MINUS_INF_CONSTANT_NORMALIZED = -0.5

def init_distribution(amino_acids=None, initial_means=None, initial_stds=None, discrete_distribution=False):
    amino_acids_list = list(amino_acids)
    if not discrete_distribution:
        components_d  = list()
        for i in range(len(initial_means)):
            mean_value = initial_means[i]
            std_value = initial_stds[i]
            d = NormalDistribution(mean_value, std_value)
            components_d.append(d)
        total_d = IndependentComponentsDistribution(components_d)
    else:
        d_dict = {key: prob for key, prob in zip(amino_acids_list, np.random.default_rng().uniform(0, 1, size=len(amino_acids_list)))}
        d_sum = sum(d_dict.values())
        d_dict = {key: value/d_sum for key, value in d_dict.items()}
        total_d = DiscreteDistribution(d_dict)
    return total_d


def init_grouped_structure_model_joiners_and_cycles_here(
        states_per_group=5, # may be int or list of numbers for each group
        number_of_groups=5,
        amino_acids="ACDEFGHIKLMNPQRSTVWY",
        freeze_distr=False,
        CYCLE_PSEUDOCOUNT=1,
        initial_params=None,
        start_cycle=True,
        end_cycle=True,
        self_cycles=False,
        discrete_distribution=False
):

    INTRAGROUP_WEIGHT = 1
    INTERGROUP_WEIGHT = 1

    START_WEIGHT = 1
    TO_CYCLE_WEIGHT = 3
    IN_CYCLE_WEIGHT = 5
    if start_cycle or end_cycle:
        TO_JOINER_WEIGHT = 5 #just to make transitions even inside groups
    else:
        TO_JOINER_WEIGHT = 1
    END_WEIGHT = 1
    FROM_CYCLE_WEIGHT = 1
    amino_acids_list = list(amino_acids)

    joiner_positions = {0: [0], 1: [1]}
    number_of_joiners = 0
    number_of_groups = number_of_groups

    if type(states_per_group) == list:
        assert number_of_groups == len(states_per_group), f"Number of groups is expected to be {number_of_groups} but got {len(states_per_group)} in a list of states per group"
    else:
        states_per_group = [states_per_group for i in range(number_of_groups)]
    distributions = list()


    mean_values = list()
    stds = list()
    if initial_params:
        for initial_distr_params in initial_params:
            mean_values.append(initial_distr_params['mean'])
            stds.append(initial_distr_params['std'])
        print(f"mean values : {mean_values}")
        print(f"std values : {stds}")

    # initialize group states
    for group_number in range(number_of_groups):
        states_in_group = states_per_group[group_number]
        for state_num in range(states_in_group):
            total_d = init_distribution(amino_acids=amino_acids,
                                        initial_means=mean_values,
                                        initial_stds=stds,
                                        discrete_distribution=discrete_distribution)
            distributions.append(total_d)
    #initialize joiner states (random here)
    joiner_label_to_index_in_model = dict()
    for joiner_name in range(number_of_joiners):
        # d = DiscreteDistribution(
        #     {key: prob for key, prob in zip(amino_acids_list, np.random.uniform(0, 1, size=len(amino_acids_list)))})
        total_d = init_distribution(amino_acids=amino_acids,
                                    initial_means=mean_values,
                                    initial_stds=stds,
                                    discrete_distribution=discrete_distribution)
        joiner_label_to_index_in_model[joiner_name] = len(distributions)
        distributions.append(total_d)
    # initialize cycle states
    if start_cycle:
        start_d = init_distribution(amino_acids=amino_acids,
                                    initial_means=mean_values,
                                    initial_stds=stds,
                                    discrete_distribution=discrete_distribution)
        if freeze_distr:
            start_d.freeze()
        distributions.append(start_d)
    if end_cycle:
        end_d = init_distribution(amino_acids=amino_acids,
                                  initial_means=mean_values,
                                    initial_stds=stds,
                                    discrete_distribution=discrete_distribution)
        if freeze_distr:
            end_d.freeze()
        distributions.append(end_d)


    # print("joiners positions", joiner_label_to_index_in_model)

    num_states = sum(states_per_group) + number_of_joiners +  int(start_cycle) + int(end_cycle) # groups, joiners, 2 cycles
    assert num_states == len(distributions)
    print(f"states will be groups:{sum(states_per_group)} + joiners:{number_of_joiners} + start/end cycles:{ int(start_cycle) + int(end_cycle)}")

    state_names = ["s{:03d}".format(i) for i in range(num_states)]
    transition_matrix = numpy.zeros((num_states, num_states))

    # start probabilities
    # connect start cycle to the first joiners
    start_cycle_idx = sum(states_per_group) + number_of_joiners
    end_cycle_idx = start_cycle_idx + 1

    # start only for first cycle and first joiner
    start_probabilities = [START_WEIGHT
                           if i < states_in_group
                           else 0
                           for i in range(num_states)]


    # do not restrict ending in multiple states, so probs are equal to transitions inside

    end_probabilities = [END_WEIGHT
                         if i > sum(states_per_group[:number_of_groups -1])
                         else 0
                         for i in range(num_states)]

    #  start cycle transitions
    if start_cycle:
        for joiner_idx in first_joiners:
            transition_matrix[start_cycle_idx][joiner_idx] = FROM_CYCLE_WEIGHT # from cycle to joiners
        transition_matrix[start_cycle_idx][start_cycle_idx] = IN_CYCLE_WEIGHT #cycle
        start_probabilities[start_cycle_idx] = TO_CYCLE_WEIGHT #from start to cycle

    # connect last joiners to last cycle
    if end_cycle:
        for joiner_idx in last_joiners:
            transition_matrix[joiner_idx][end_cycle_idx] = TO_CYCLE_WEIGHT
        transition_matrix[end_cycle_idx][end_cycle_idx] = IN_CYCLE_WEIGHT


    # Transitions inside groups
    for group_number in range(number_of_groups):
        states_in_group = states_per_group[group_number]
        for i in range(states_in_group):
            state_num_from = sum(states_per_group[:group_number]) + i
            # fill transitions inside each group with INTRAGROUP_WEIGHT
            for j in range(states_in_group):

                state_num_to =  sum(states_per_group[:group_number]) + j
                if state_num_from != state_num_to:
                    transition_matrix[state_num_from][state_num_to] = INTRAGROUP_WEIGHT
            if self_cycles:
                transition_matrix[state_num_from][state_num_from] = INTRAGROUP_WEIGHT
            # transition to next group
            if group_number != number_of_groups - 1:
                for j in range(states_in_group):
                    state_num_to = sum(states_per_group[:group_number+1]) + j
                    transition_matrix[state_num_from][state_num_to] = INTERGROUP_WEIGHT



    # # connect first joiner to first group
    # for joiner_index in first_joiners:
    #     group_number = 0
    #     state_num_from = joiner_index
    #     states_in_group =  states_per_group[group_number]
    #     for j in range(states_in_group):
    #         state_num_to = sum(states_per_group[:group_number]) + j
    #         transition_matrix[state_num_from][state_num_to] = INTERGROUP_WEIGHT

    # # connect last group to the last joiner
    # for joiner_index in last_joiners:
    #     group_number = number_of_groups -1
    #     state_num_to = joiner_index
    #     states_in_group =  states_per_group[group_number]
    #     for j in range(states_in_group):
    #         state_num_from = sum(states_per_group[:group_number]) + j
    #         transition_matrix[state_num_from][state_num_to] = TO_JOINER_WEIGHT
    model_name = 'grouped_model'


    # assert sum(start_probabilities) == len(joiner_positions[0])* START_WEIGHT + TO_CYCLE_WEIGHT, start_probabilities
    # assert sum(end_probabilities) ==  (len(joiner_positions[max(joiner_positions.keys())]) + 1)  * END_WEIGHT, end_probabilities + list(last_joiners)

    model = HiddenMarkovModel.from_matrix(
        name=model_name,
        transition_probabilities=transition_matrix,
        starts=start_probabilities,
        distributions=distributions,
        ends=end_probabilities,
        state_names=state_names,
        verbose=True)
    # Update created model with specific pseudocounts
    if start_cycle:
        model.add_transition(model.states[start_cycle_idx], model.states[start_cycle_idx], IN_CYCLE_WEIGHT, pseudocount=CYCLE_PSEUDOCOUNT)
        model.add_transition(model.start, model.states[start_cycle_idx], IN_CYCLE_WEIGHT, pseudocount=CYCLE_PSEUDOCOUNT)
        for joiner_idx in first_joiners:
            model.add_transition(model.states[start_cycle_idx], model.states[joiner_idx], IN_CYCLE_WEIGHT, pseudocount=CYCLE_PSEUDOCOUNT)
    if end_cycle:
        model.add_transition(model.states[end_cycle_idx], model.states[end_cycle_idx], IN_CYCLE_WEIGHT, pseudocount=CYCLE_PSEUDOCOUNT)
        model.add_transition(model.states[end_cycle_idx],model.end, IN_CYCLE_WEIGHT, pseudocount=CYCLE_PSEUDOCOUNT)
        for joiner_idx in last_joiners:
            model.add_transition(model.states[joiner_idx], model.states[end_cycle_idx], IN_CYCLE_WEIGHT, pseudocount=CYCLE_PSEUDOCOUNT)


        # model.add_transition(model.states[end_cycle_idx], model.states[end_cycle_idx], IN_CYCLE_WEIGHT, pseudocount=CYCLE_PSEUDOCOUNT)

    model.bake(verbose=True)
    return model

In [76]:

model = init_grouped_structure_model_joiners_and_cycles_here(
    states_per_group=int(params['states_per_group']),
    number_of_groups=params['groups'],
    freeze_distr= params['freeze_distr'],
    CYCLE_PSEUDOCOUNT=params['cycle_pseudocount'] ,
    initial_params=None,
    start_cycle=params['start_cycle'],
    end_cycle=params['end_cycle'],
    self_cycles=params['self_cycle'],
    discrete_distribution=check_peptides)


states will be groups:20 + joiners:0 + start/end cycles:0
grouped_model : grouped_model-start summed to 5.0, normalized to 1.0
grouped_model : s000 summed to 9.0, normalized to 1.0
grouped_model : s001 summed to 9.0, normalized to 1.0
grouped_model : s002 summed to 9.0, normalized to 1.0
grouped_model : s003 summed to 9.0, normalized to 1.0
grouped_model : s004 summed to 9.0, normalized to 1.0
grouped_model : s005 summed to 9.0, normalized to 1.0
grouped_model : s006 summed to 9.0, normalized to 1.0
grouped_model : s007 summed to 9.0, normalized to 1.0
grouped_model : s008 summed to 9.0, normalized to 1.0
grouped_model : s009 summed to 9.0, normalized to 1.0
grouped_model : s010 summed to 9.0, normalized to 1.0
grouped_model : s011 summed to 9.0, normalized to 1.0
grouped_model : s012 summed to 9.0, normalized to 1.0
grouped_model : s013 summed to 9.0, normalized to 1.0
grouped_model : s014 summed to 9.0, normalized to 1.0
grouped_model : s015 summed to 4.0, normalized to 1.0
grouped_m

In [77]:
with open('model.png', 'w+') as f:
    model.plot(file=f, crop_zero=True, rotation=90, rotate=90)

In [78]:
from hmm_visualization_methods import *
import seaborn as sns
sns.set_style("white")
plot_distributions_for_states(model,"test/", horizontal=False, discrete=check_peptides, initial_params=initial_params, matrix_type='Prob')

In [79]:
pv = make_pyviz_graph(model, "test/", precision=3)

Converting graph for model (grouped_model) (Delete zero edges)


## Finaly prepare models

In [80]:
per_allele_per_run_per_split_prepared_models = dict()
for allele_name in ALLELES:
    target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_').replace('/', '_')
    per_allele_per_run_per_split_prepared_models[allele_name] = dict()
    for run_num in range(RUNS):
        per_allele_per_run_per_split_prepared_models[allele_name][run_num] = dict()
        for split_num in range(TOTAL_SPLITS):
            # prepared_model  = init_grouped_structure_model(states_per_group=int(params['st']/params['groups']), number_of_groups=params['groups'])
            prepared_model = init_grouped_structure_model_joiners_and_cycles_here(
                states_per_group=int(params['states_per_group']),
                number_of_groups=params['groups'],
                CYCLE_PSEUDOCOUNT=params['cycle_pseudocount'],
                freeze_distr=params['freeze_distr'],
                initial_params=initial_params,
                start_cycle=params['start_cycle'],
                end_cycle=params['end_cycle'],
                self_cycles=params['self_cycle'],
                discrete_distribution=check_peptides
            )


            prepared_model.name = f'{run_num}_run-{lengths_string}-{params_string}_model-{target_allele_name}-{split_num}'
            per_allele_per_run_per_split_prepared_models[allele_name][run_num][split_num] = prepared_model

states will be groups:20 + joiners:0 + start/end cycles:0
grouped_model : grouped_model-start summed to 5.0, normalized to 1.0
grouped_model : s000 summed to 9.0, normalized to 1.0
grouped_model : s001 summed to 9.0, normalized to 1.0
grouped_model : s002 summed to 9.0, normalized to 1.0
grouped_model : s003 summed to 9.0, normalized to 1.0
grouped_model : s004 summed to 9.0, normalized to 1.0
grouped_model : s005 summed to 9.0, normalized to 1.0
grouped_model : s006 summed to 9.0, normalized to 1.0
grouped_model : s007 summed to 9.0, normalized to 1.0
grouped_model : s008 summed to 9.0, normalized to 1.0
grouped_model : s009 summed to 9.0, normalized to 1.0
grouped_model : s010 summed to 9.0, normalized to 1.0
grouped_model : s011 summed to 9.0, normalized to 1.0
grouped_model : s012 summed to 9.0, normalized to 1.0
grouped_model : s013 summed to 9.0, normalized to 1.0
grouped_model : s014 summed to 9.0, normalized to 1.0
grouped_model : s015 summed to 4.0, normalized to 1.0
grouped_m

## Do training

In [81]:
from hmm_logic_methods import train_model, save_model, add_more_states_and_reset_transitions, init_grouped_structure_model_joiners_and_cycles


In [82]:
%%time

per_name_models = dict()
per_allele_per_run_per_split_new_histories = dict()
for allele_name in ALLELES:
    per_allele_per_run_per_split_new_histories[allele_name] = dict()
    for run_num in range(RUNS):
        print(f"run {run_num}")
        per_kfold_per_length_data = per_allele_per_kfold_per_length_binders_train[allele_name]
        per_kfold_per_length_weights = per_allele_per_kfold_per_length_weights[allele_name]


        models, histories = train_model(allele_name=allele_name,
                                        per_kfold_per_length_data=per_kfold_per_length_data,
                                        per_kfold_per_length_weights_for_data=per_kfold_per_length_weights,
                                        target_lengths=TARGET_LENGTHS,
                                        multiple_check_input=params['multiple_check_input'],
                                        algorithm=params['alg'],
                                        verbose=params['verbose'],
                                        per_kfold_per_length_test_data=per_allele_per_kfold_per_length_binders_test[allele_name],
                                        stop_threshold=params['stop_threshold'],
                                        min_iterations= params['min_iters'],
                                        max_iterations=params['max_iters'],
                                        prepared_models=per_allele_per_run_per_split_prepared_models[allele_name][run_num],
                                        emission_pseudocount=params['emission_pseudocount'],
                                        transition_pseudocount=params['transition_pseudocount'],
                                        edge_inertia=params['edge_inertia'],
                                        distribution_inertia=params['distribution_inertia'],
                                        lr_decay=params['lr_decay'],
                                        use_pseudocount=params['use_pseudocount'])

        per_allele_per_run_per_split_prepared_models[allele_name][run_num] = models
        per_allele_per_run_per_split_new_histories[allele_name][run_num] = histories
        # per_allele_per_run_per_split_models[allele_name][run_num] = models
        for split_num, model in models.items():
            per_name_models[model.name] = model
        # save models
        for split_num, model in models.items():
            history = histories[split_num]
            save_model(TARGET_PATH_TO_RESULTS, model,history)



run 0
split 0
train size 14143
Shuffling for fairness 
Simple training: Train Length 14143, init_batch_size None
Use pseudocount is False
[1] Improvement: 31073.462312543124	Time (s): 1.064
[2] Improvement: 2275.5946504795575	Time (s): 1.091
[3] Improvement: -132.20082904514857	Time (s): 1.091
[4] Improvement: -65.80787355074426	Time (s): 0.9579
[5] Improvement: 163.6623731290456	Time (s): 1.098
[6] Improvement: 191.67900146142347	Time (s): 1.539
[7] Improvement: 214.3215247487533	Time (s): 2.25
[8] Improvement: 187.66438406181987	Time (s): 1.659
[9] Improvement: 100.60083869087975	Time (s): 1.311
[10] Improvement: 109.64655160321854	Time (s): 1.257
[11] Improvement: 120.4368748095003	Time (s): 1.293
[12] Improvement: 333.8272380112321	Time (s): 1.404
[13] Improvement: 195.16181558766402	Time (s): 1.288
[14] Improvement: 244.06688342237612	Time (s): 1.764
[15] Improvement: 275.67899449612014	Time (s): 1.294
[16] Improvement: 310.2126669156132	Time (s): 1.382
[17] Improvement: 315.02904

In [83]:
from hmm_visualization_methods import *

In [84]:
%%time

for i, model in enumerate(per_name_models.values()):
    print(i)
    # with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph.png", 'w+') as f:
    #     model.plot(file=f, crop_zero=False)
    with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph_cropped.png", 'w+') as f:
        model.plot(file=f, crop_zero=True)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
CPU times: total: 312 ms
Wall time: 7.9 s


In [85]:
%%time

for i, model in enumerate(per_name_models.values()):
    plot_distributions_for_states(model,TARGET_PATH_TO_RESULTS, horizontal=False, discrete=check_peptides, initial_params=initial_params)
    print(i)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
CPU times: total: 1min 17s
Wall time: 1min 19s


In [86]:
from hmm_visualization_methods import *


per_name_pyviz = dict()
for name, model in per_name_models.items():
    per_name_pyviz[name] = make_pyviz_graph(model, TARGET_PATH_TO_RESULTS, precision=3)

Converting graph for model (0_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0) (Delete zero edges)
Converting graph for model (1_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0) (Delete zero edges)
Converting graph for model (2_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0) (Delete zero edges)
Converting graph for model (3_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0) (Delete zero edges)
Converting graph for model (4_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0) (Delete zero edges)
Converting graph for model (5_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0) (Delete zero edges)
Converting graph for model (6_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0) (Delete zero edges)
Converting graph for model (7_run-9-20-st